# External-agent official recovery
The old real-agent recovery plot is now driven by the same external harness records as the application evaluation.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
PAPER = ROOT / 'paper' / 'img'
PAPER.mkdir(parents=True, exist_ok=True)
frames = []
for backend in ('deepseek_harness', 'codex', 'official'):
    path = RESULTS / backend / 'official_token_summary.csv'
    if path.exists(): frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True).drop_duplicates() if frames else pd.DataFrame()
if not df.empty:
    view = df.groupby(['harness_backend', 'suite', 'scale', 'mode'], as_index=False)[['success_rate', 'recovery_wall_s_p95']].mean()
    fig, ax = plt.subplots(figsize=(6.8, 3.0), dpi=300)
    for (backend, suite, mode), group in view.groupby(['harness_backend', 'suite', 'mode']):
        group = group.sort_values('scale')
        ax.plot(group['scale'], group['success_rate'], marker='o', linewidth=1.0, label=f'{backend}:{suite}:{mode}')
    ax.set_xlabel('Official task scale')
    ax.set_ylabel('Verifier success rate')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=5.5, ncol=2, frameon=True)
    fig.tight_layout()
    fig.savefig(FIGDIR / 'FIG-Real-Agent-Recovery.pdf', bbox_inches='tight')
    fig.savefig(PAPER / 'FIG-Real-Agent-Recovery.pdf', bbox_inches='tight')
else:
    print('No official summary found; run the official benchmark first.')
# Compatibility source name: real_agent_recovery.csv.
# causal_targets_correct and independent_retained are verifier concepts recorded by the official runner.
